# Fine-tune Qwen2.5-1.5B-Instruct voi TinyLoRA (PEFT)

Notebook nay huong dan su dung **TinyLoRA** de fine-tune `Qwen/Qwen2.5-1.5B-Instruct`
cho tap du lieu **ViMMRC 2.0** (doc hieu trac nghiem tieng Viet).

## TinyLoRA vs LoRA thuong:
| Dac diem | LoRA | TinyLoRA |
|---|---|---|
| Tham so huan luyen | `r * (d_in + d_out)` moi layer | Chi vector `v` kich thuoc `u` |
| Co che | Hoc 2 ma tran A, B | Tong co trong so cac phep chieu ngau nhien co dinh |
| Hieu qua bo nho | Tot | Cuc tot (~13 tham so co the du) |
| Phu hop | Fine-tuning chuan | RL / tai nguyen rat han che |

In [ ]:
# Buoc 1: Go cai dat cac thu vien cu de tranh xung dot
!pip uninstall torch torchvision torchaudio xformers transformers trl unsloth unsloth_zoo -y

In [ ]:
# Buoc 2: Cai dat lai PyTorch voi CUDA 12.8
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --no-cache-dir

In [ ]:
# Buoc 3: Cai dat cac thu vien can thiet
# peft >= 0.14.0 ho tro TinyLoraConfig
!pip install transformers trl peft accelerate bitsandbytes xformers --no-cache-dir

# Buoc 4: Cai dat Unsloth moi nhat de load model nhanh
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --no-cache-dir

In [ ]:
import torch
import peft
import transformers

print("--- KIEM TRA PHIEN BAN CAC THU VIEN ---")
print(f"PyTorch version:      {torch.__version__}")
print(f"CUDA Available:       {torch.cuda.is_available()}")
print(f"PEFT version:         {peft.__version__}")
print(f"Transformers version: {transformers.__version__}")

try:
    from peft import TinyLoraConfig
    print("TinyLoraConfig: OK (available)")
except ImportError:
    print("TinyLoraConfig: NOT FOUND - Can peft >= 0.14.0")
    print("Chay: pip install --upgrade peft")

In [ ]:
# ============================================================
# 2. Load Model va Tokenizer voi Unsloth (tang toc 2x)
# ============================================================
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Ho tro tu dong RoPE Scaling
dtype          = None  # Tu dong nhan dien (Float16/Bfloat16)
load_in_4bit   = True  # 4-bit quantization de tiet kiem GPU RAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = "Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype          = dtype,
    load_in_4bit   = load_in_4bit,
)

print("Model va Tokenizer da duoc load thanh cong!")

In [ ]:
# ============================================================
# 3. AP DUNG TINYLORA (thay the LoRA thuong)
# ============================================================
from peft import TinyLoraConfig, get_peft_model

# Giai thich tham so TinyLoRA:
# ---------------------------------------------------------------
# r               : Rank cua khong gian con (giong LoRA)
# num_projections : So luong phep chieu ngau nhien (u)
#                   -> Quyet dinh so tham so huan luyen moi layer
#                   -> Nho hon = it tham so hon (extreme efficiency)
# weight_tying    : 0.0 = moi layer co vector v doc lap
#                   1.0 = tat ca layer dung chung 1 vector
# ---------------------------------------------------------------
tinylora_config = TinyLoraConfig(
    r = 16,                  # Rank cua low-rank decomposition
    num_projections = 4,     # So phep chieu ngau nhien (u), so tham so = u moi layer
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0.0,      # Nen de 0 voi TinyLoRA
    bias = "none",
    weight_tying = 0.0,      # 0.0: moi layer co vector v rieng
    task_type = "CAUSAL_LM",
)

# Lay model goc tu Unsloth wrapper
base_model = model.model if hasattr(model, "model") else model

# Ap dung TinyLoRA len model
model = get_peft_model(base_model, tinylora_config)
model.print_trainable_parameters()
print("\nTinyLoRA da duoc ap dung thanh cong!")

In [ ]:
# ============================================================
# PHUONG AN DU PHONG: Neu TinyLoraConfig chua co trong PEFT,
# dung LoRA rank rat nho (r=2) + rslora de mo phong TinyLoRA style
# Bo comment cell nay va comment cell tren khi can
# ============================================================

# from unsloth import FastLanguageModel
# model = FastLanguageModel.get_peft_model(
#     model,
#     r = 2,                         # Rank rat nho - TinyLoRA style
#     target_modules = [
#         "q_proj", "k_proj", "v_proj", "o_proj",
#         "gate_proj", "up_proj", "down_proj",
#     ],
#     lora_alpha = 4,
#     lora_dropout = 0,
#     bias = "none",
#     use_gradient_checkpointing = "unsloth",
#     random_state = 3407,
#     use_rslora = True,             # Rank-stabilized LoRA
#     loftq_config = None,
# )
# model.print_trainable_parameters()

In [ ]:
# ============================================================
# 4. Tien xu ly du lieu ViMMRC 2.0
# ============================================================
import json
from datasets import Dataset

# Tai du lieu (dam bao train_vimmrc.json nam cung thu muc notebook)
with open("train_vimmrc.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

formatted_data = []

for item in raw_data:
    article      = item.get("article", "")
    questions    = item.get("questions", [])
    options_list = item.get("options", [])
    answers      = item.get("answers", [])

    for q, opts, ans in zip(questions, options_list, answers):
        letters = ["A", "B", "C", "D", "E", "F", "G", "H"]
        options_text = ""
        for i, opt in enumerate(opts):
            if i < len(letters):
                options_text += f"{letters[i]}. {opt}\n"

        prompt = (
            f"Doc doan van sau va tra loi cau hoi bang cach chon dap an dung nhat.\n\n"
            f"Doan van:\n{article}\n\n"
            f"Cau hoi: {q}\n"
            f"Cac dap an:\n{options_text}"
        )
        response = f"{ans}"

        formatted_data.append({
            "instruction": prompt,
            "output": response
        })

dataset = Dataset.from_list(formatted_data)
print(f"Tong so mau huan luyen: {len(dataset)}")
print("\nVi du 1 mau:\n", dataset[0])

In [ ]:
# ============================================================
# 5. Format prompt theo chuan ChatML (Qwen2.5 Instruct)
# ============================================================
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        messages = [
            {
                "role": "system",
                "content": "Ban la mot tro ly AI thong minh, gioi tieng Viet va co kha nang doc hieu van ban cuc ky chinh xac."
            },
            {"role": "user",      "content": instruction},
            {"role": "assistant", "content": output}
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print("Dataset da duoc format theo ChatML.")
print("\nVi du text sau format:\n", dataset[0]["text"][:500])

In [ ]:
# ============================================================
# 6. Cau hinh va Huan luyen voi TinyLoRA
# ============================================================
from trl import SFTTrainer
from transformers import TrainingArguments
import sys

# Tu dong chon fp16/bf16 dua theo GPU
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = not use_bf16

trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = dataset,
    dataset_text_field = "text",
    max_seq_length   = max_seq_length,
    dataset_num_proc = 1,    # Ep ve 1 de tranh loi pickle
    packing          = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps                = 5,
        # max_steps = 60,            # Bo comment de chay thu nhanh
        num_train_epochs            = 3,
        learning_rate               = 2e-4,  # TinyLoRA hoat dong tot voi LR cao
        fp16                        = use_fp16,
        bf16                        = use_bf16,
        logging_steps               = 10,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "cosine",
        seed                        = 3407,
        output_dir                  = "outputs_tinylora",
        save_strategy               = "epoch",
        report_to                   = "none",
    ),
)

# Fix compatibility giua cac phien ban trl khac nhau
real_config_cls  = type(trainer.args)
real_trainer_cls = type(trainer)
sys.modules['trl.trainer.sft_config']  = sys.modules[real_config_cls.__module__]
sys.modules['trl.trainer.sft_trainer'] = sys.modules[real_trainer_cls.__module__]
sys.modules[real_config_cls.__module__].SFTConfig   = real_config_cls
sys.modules[real_trainer_cls.__module__].SFTTrainer = real_trainer_cls

print("Bat dau training voi TinyLoRA...")
model.print_trainable_parameters()

trainer_stats = trainer.train()
print(f"\nTraining hoan tat! Thoi gian: {trainer_stats.metrics.get('train_runtime', 0):.2f} giay")

In [ ]:
# ============================================================
# 7. Kiem tra Inference sau huan luyen
# ============================================================
model.eval()

test_instruction = dataset[0]["instruction"]

messages = [
    {
        "role": "system",
        "content": "Ban la mot tro ly AI thong minh, gioi tieng Viet va co kha nang doc hieu van ban cuc ky chinh xac."
    },
    {"role": "user", "content": test_instruction},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize             = True,
    add_generation_prompt = True,
    return_tensors       = "pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        input_ids      = inputs,
        max_new_tokens = 64,
        use_cache      = True,
        temperature    = 0.1,
        do_sample      = True,
    )

print("\n--- KET QUA SINH TU MODEL DA FINE-TUNE (TinyLoRA) ---\n")
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

In [ ]:
# ============================================================
# 8. Luu Model TinyLoRA
# ============================================================
import os

save_path = "qwen2.5-1.5b-instruct-tinylora-vinmmrc"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Da luu thanh cong mo hinh TinyLoRA tai: {save_path}")
print("\nFile duoc luu bao gom:")
for f in os.listdir(save_path):
    size = os.path.getsize(os.path.join(save_path, f))
    print(f"  - {f}: {size / 1024:.1f} KB")

# Neu muon merge TinyLoRA vao model goc (merge va unload adapter):
# merged = model.merge_and_unload()
# merged.save_pretrained("qwen2.5-1.5b-instruct-tinylora-merged")
# tokenizer.save_pretrained("qwen2.5-1.5b-instruct-tinylora-merged")

In [ ]:
# ============================================================
# 9. Danh gia tren tap Test (Evaluation)
# ============================================================
import json
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from sklearn.metrics import accuracy_score, classification_report

# ------------------------------------------
# 1. CAU HINH & LOAD MODEL
# ------------------------------------------
base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_path    = "qwen2.5-1.5b-instruct-tinylora-vinmmrc"
test_data_path  = "test_vimmrc.json"

print("Loading Tokenizer va Base Model...")
eval_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
eval_base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype = torch.float16,
    device_map  = "auto"
)
eval_model = PeftModel.from_pretrained(eval_base_model, adapter_path)
eval_model.eval()
print("Model da duoc load thanh cong!")

with open(test_data_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

# ------------------------------------------
# 2. HAM TAO PROMPT
# ------------------------------------------
def format_prompt(article, question, options):
    labels = ["A", "B", "C", "D", "E", "F"]
    options_text = "\n".join([f"{labels[i]}. {opt}" for i, opt in enumerate(options)])
    prompt = (
        f"Dua vao doan van sau, hay chon dap an dung nhat cho cau hoi.\n\n"
        f"Doan van: {article}\n\n"
        f"Cau hoi: {question}\n"
        f"Cac lua chon:\n{options_text}\n\n"
        f"Dap an cua ban (chi dua ra mot chu cai): "
    )
    messages = [
        {"role": "system", "content": "Ban la mot tro ly AI gioi tieng Viet, co kha nang doc hieu va tra loi cau hoi trac nghiem chinh xac."},
        {"role": "user", "content": prompt}
    ]
    return eval_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# ------------------------------------------
# 3. CHAY INFERENCE TREN TOAN BO TAP TEST
# ------------------------------------------
y_true, y_pred = [], []
invalid_preds  = 0
all_samples    = []

for item in test_data:
    article = item["article"]
    for i in range(len(item["questions"])):
        if len(item["options"][i]) > 0:
            all_samples.append({
                "article":  article,
                "question": item["questions"][i],
                "options":  item["options"][i],
                "answer":   item["answers"][i]
            })

print(f"\nTien hanh test tren {len(all_samples)} cau hoi...")

for sample in tqdm(all_samples):
    prompt = format_prompt(sample["article"], sample["question"], sample["options"])
    inputs = eval_tokenizer(prompt, return_tensors="pt").to(eval_model.device)
    with torch.no_grad():
        outputs = eval_model.generate(
            **inputs,
            max_new_tokens = 3,
            pad_token_id   = eval_tokenizer.pad_token_id,
            eos_token_id   = eval_tokenizer.eos_token_id,
            temperature    = 0.01,
            do_sample      = True,
        )
    input_length  = inputs.input_ids.shape[1]
    response      = eval_tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()
    pred_label    = response[0].upper() if len(response) > 0 else "N/A"
    true_label    = sample["answer"].upper()
    y_true.append(true_label)
    if pred_label in ["A", "B", "C", "D"]:
        y_pred.append(pred_label)
    else:
        y_pred.append("X")
        invalid_preds += 1

# ------------------------------------------
# 4. TINH TOAN METRICS
# ------------------------------------------
print("\n" + "="*55)
print("KET QUA DANH GIA (EVALUATION METRICS) - TinyLoRA")
print("="*55)
accuracy = accuracy_score(y_true, y_pred)
print(f"1. Tong so cau hoi:                   {len(y_true)}")
print(f"2. Cau tra loi dung dinh dang:         {len(y_true) - invalid_preds}")
print(f"3. Cau bi loi dinh dang (khong A-D):   {invalid_preds}")
print(f"4. OVERALL ACCURACY:                   {accuracy * 100:.2f}%")
print("\n5. CLASSIFICATION REPORT:")
print(classification_report(y_true, y_pred, zero_division=0))